In [16]:
# example of bayesian optimization for a 1d function from scratch
from math import sin
from math import pi
from numpy import arange
from numpy import vstack
from numpy import argmax
from numpy import asarray
from numpy.random import normal
from numpy.random import random
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from warnings import catch_warnings
from warnings import simplefilter
# from matplotlib import pyplot

import numpy as np
import scipy.stats as sps
from sklearn.datasets import load_iris
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern


from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

In [3]:
# 创建形状为 (6, 2) 的 NumPy 数组 bounds
bounds = np.array([[1, 100], [1, 100], [1, 100], [1, 100], [1, 100], [1, 100]])
count = 0

In [4]:
# objective function
def objective(params):    
	global count  # Declare count as global   
	count = count + 1
	return sum(params)

In [5]:
# surrogate or approximation for the objective function
def surrogate(model, X):
	# catch any warning generated when making a prediction
	with catch_warnings():
		# ignore generated warnings
		simplefilter("ignore")
		return model.predict(X, return_std=True)

In [19]:
# probability of improvement acquisition function
def acquisition(Xsamples, model):
	# calculate the best surrogate score found so far
	mu, sigma = surrogate(model, Xsamples)
	f_best = np.max(mu)
	probs = norm.cdf((mu - f_best) / (sigma+1E-9))
	return probs


	
	# improvement = f_best - mu
	# with np.errstate(divide='warn'):
	# 	Z = improvement / sigma if sigma > 0 else 0
	# 	ei = improvement * sps.norm.cdf(Z) + sigma * sps.norm.pdf(Z)
	# 	ei[sigma == 0.0] == 0.0
	# return ei

In [10]:
def opt_acquisition(X, y, model):
	global bounds
	# random search, generate random samples
	Xsamples = np.random.uniform(low=bounds[:, 0], high=bounds[:, 1], size=(100, 6))
	# Xsamples = random(100)
	# Xsamples = Xsamples.reshape(len(Xsamples), 1)
	# calculate the acquisition function for each sample
	scores = acquisition(Xsamples, model)
	# locate the index of the largest scores
	ix = argmax(scores)
	returnX = Xsamples[ix]
	return returnX

In [17]:
# sample the domain sparsely with noise
# X = random(100)
X = np.random.uniform(low=bounds[:, 0], high=bounds[:, 1], size=(20, 6))
print(X)

y = np.random.uniform(6, 600, size=X.shape[0])  # Generate one value per row in X
print(y)
# y = asarray([objective(x) for x in X])
# reshape into rows and cols
# X = X.reshape(len(X), 1)
# y = y.reshape(len(y), 1)
# define the model
# 定义核函数
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))
# 创建高斯过程回归模型
model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
# model = GaussianProcessRegressor()
# fit the model
model.fit(X, y)
# plot before hand
# plot(X, y, model)

[[ 9.81267837 71.32214364 42.25289289 99.03042313 22.00780117 71.26910657]
 [ 8.78537478 37.59499673  6.34127605 16.73187518 18.4467773  98.69761381]
 [81.47135876 46.34312148 40.57342168 81.02746269  4.36772518 92.12418477]
 [20.08879153  6.35847453 16.56750643 62.68834901 95.1486974  59.25592742]
 [98.65794394 44.56760897 83.6081546  43.6977135  79.37468239 96.79254372]
 [ 9.40826741 53.43203651 55.36146097 11.0971208  12.60111654 55.3523523 ]
 [20.7231116  62.27926968 56.30099927  6.0370243  27.25194792 93.60815267]
 [34.44191179 27.07153175 48.58504297 14.92162485 96.56132926 22.80594806]
 [63.6168037  99.41555306 32.32808344 77.0523037  22.09888533 21.73378548]
 [30.37641578 83.56592993 66.98891627 64.32988059 58.12287882 23.16445135]
 [91.48632048 79.12209502 50.26252552 57.83642643 69.34944837 57.64051303]
 [ 4.38653583 37.57231073 90.00067395 61.59973022  1.5121277   6.73879369]
 [47.00055807  6.57050128 39.52095272  3.18374655 37.81630055 49.02317862]
 [58.90139947 92.44143933

c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


GaussianProcessRegressor(kernel=1**2 * RBF(length_scale=1),
                         n_restarts_optimizer=10)

In [21]:

# perform the optimization process
for i in range(100):
	# select the next point to sample
	X_next = opt_acquisition(X, y, model)
	# sample the point
	next_y = objective(X_next)
	# summarize the finding
	# est, _ = surrogate(model, [[x]])
	print('actual=%.3f' % (next_y))
	# add the data to the dataset
	X = vstack((X, X_next))
	# y = vstack(y, actual)
	y = np.append(y, next_y)
	# update the model
	model.fit(X, y)

actual=304.622
actual=333.920
actual=305.328


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaus

actual=303.960
actual=198.018
actual=312.899
actual=287.860


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


actual=166.039
actual=295.455
actual=359.055


...

c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


actual=398.692
actual=361.963
actual=181.122


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


actual=364.010
actual=363.677
actual=361.206


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
